# MPECSS - MPECLib Benchmark

This notebook runs the **MPECLib** benchmark suite.

- **Dataset**: MPECLib
- **Problems**: 92
- **Estimated time**: 4-6 hours
- **Resume support**: Built-in

Run the cells in order. After a Kaggle restart, re-run cells 1-3, then use the **Resume** cell.

## 1. Configuration

In [ ]:
# ┌─────────────────────────────────────────────────┐
# │  MPECLib Configuration                           │
# └─────────────────────────────────────────────────┘
DATASET = 'mpeclib'
RUN_TAG = 'Kaggle_MPECLib'

WORKERS = 4
TIMEOUT = 3600
SAVE_LOGS = True
SHUFFLE = True

# Paths
import os
REPO_DIR = "/kaggle/working/MPECSSCODEPAPER"
REPO_GIT_URL = "https://github.com/mrsaurabhtanwar/MPECSSCODEPAPER.git"

# Dataset path (from Kaggle dataset)
KAGGLE_DATASET_PATH = '/kaggle/input/mpecss-benchmarks/benchmarks/mpeclib/mpeclib-json'
if not os.path.exists(KAGGLE_DATASET_PATH):
    KAGGLE_DATASET_PATH = None
    print("[WARNING] Kaggle dataset not found. Add 'mpecss-benchmarks' dataset to this notebook.")

## 2. Clone Repository

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_DIR = Path(REPO_DIR)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print(f"Cloning: {REPO_GIT_URL}")
subprocess.run(["git", "clone", "--depth", "1", REPO_GIT_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
print(f"[OK] Repository ready at: {REPO_DIR}")

## 3. Install Dependencies

In [ ]:
%%bash
cd /kaggle/working/MPECSSCODEPAPER
pip install -q --upgrade pip
pip install -q -e .
echo "[OK] Installation complete"

## 4. Verify Setup

In [ ]:
import os

DATASET_PATH = KAGGLE_DATASET_PATH or f"{REPO_DIR}/benchmarks/mpeclib/mpeclib-json"
print(f"Dataset path: {DATASET_PATH}")

if os.path.exists(DATASET_PATH):
    count = len([f for f in os.listdir(DATASET_PATH) if f.endswith('.json')])
    print(f"[OK] Found {count} problems")
else:
    print(f"[ERROR] Path not found: {DATASET_PATH}")

## 5. System Info

In [ ]:
import multiprocessing
import platform
import psutil

print("=" * 50)
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"CPUs: {multiprocessing.cpu_count()}")
mem = psutil.virtual_memory()
print(f"RAM: {mem.available / 1024**3:.1f} GB available / {mem.total / 1024**3:.1f} GB total")
print("=" * 50)

## 6. Preflight Checks

In [ ]:
%%bash
cd /kaggle/working/MPECSSCODEPAPER
python scripts/preflight_checks.py

## 7. Define Runner

In [ ]:
import subprocess
import sys
from pathlib import Path

def run_benchmark(resume=False, summary=False):
    cmd = [
        sys.executable,
        str(Path(REPO_DIR) / "kaggle_setup" / "resumable_benchmark.py"),
        "--dataset", DATASET,
        "--repo-dir", str(REPO_DIR),
        "--tag", RUN_TAG,
        "--workers", str(WORKERS),
        "--timeout", str(TIMEOUT),
        "--path", DATASET_PATH,
        "--skip-preflight",
    ]
    if SAVE_LOGS: cmd.append("--save-logs")
    if SHUFFLE: cmd.append("--shuffle")
    if resume: cmd.append("--resume-latest")
    if summary: cmd.append("--summary-only")
    
    print("+ " + " ".join(cmd))
    subprocess.run(cmd)

## 8. Run Benchmark (Fresh Start)

In [ ]:
run_benchmark()

## 9. Resume (After Restart)

In [ ]:
run_benchmark(resume=True)

## 10. Progress Summary

In [ ]:
run_benchmark(summary=True)

## 11. Copy Results

In [ ]:
%%bash
mkdir -p /kaggle/working/outputs
cp -r /kaggle/working/MPECSSCODEPAPER/results/* /kaggle/working/outputs/ 2>/dev/null || true
echo "[OK] Results copied to /kaggle/working/outputs/"
ls -la /kaggle/working/outputs/

## 12. Software Versions

In [ ]:
import casadi, numpy, scipy, pandas, platform
print(f"Python: {platform.python_version()}")
print(f"NumPy: {numpy.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Pandas: {pandas.__version__}")
print(f"CasADi: {casadi.__version__}")